# Quiz de treino, Avaliação 26.05

**Contexto**: Você é analista de combustíveis e recebeu dois datasets da ANP:
- `postos_sp_2025.csv`, com 76 mil coletas de preço de revenda de postos em SP (jan a jun/2025).
- `distribuicao_municipios_sp_2025.csv`, com preço de atacado (distribuição) por município, mês e produto.

Você vai responder 16 perguntas que progridem do simples ao composto. Tempo sugerido de treino: 90 minutos. Use só a folha de cola.

Cada pergunta tem uma célula de código vazia logo abaixo pra você resolver. O gabarito vem no final, mas só olha **depois** de tentar.

Roda a célula de setup abaixo primeiro.

In [ ]:
import pandas as pd

postos = pd.read_csv(
    "/Users/arthurmalucelli/FGV/Programacao/Aulas/05.26/postos_sp_2025.csv",
    sep=";", decimal=",",
)
distrib = pd.read_csv(
    "/Users/arthurmalucelli/FGV/Programacao/Aulas/05.26/distribuicao_municipios_sp_2025.csv",
    sep=";", decimal=",",
)

print("postos:", postos.shape)
print("distrib:", distrib.shape)
postos.head()

## Bloco 1, exploração inicial

### Pergunta 1
Quantas linhas e quantas colunas o DataFrame `postos` tem? E o `distrib`?

### Pergunta 2
Mostre as 10 primeiras linhas de `postos`. Confirme com `info()` que `Valor de Venda` está como `float64` (não `object`). Se estivesse como object, o que teria dado errado no setup?

### Pergunta 3
O CNPJ no arquivo vem com **um espaço no começo** de cada valor. Limpe esse espaço usando `.str.strip()` e depois conte quantos postos físicos distintos existem no dataset (CNPJs únicos).

## Bloco 2, análise por produto

### Pergunta 4
Quantas coletas existem de cada produto (GASOLINA, ETANOL, DIESEL S10)? Use `value_counts`.

### Pergunta 5
Calcule o preço médio, mediana, mínimo e máximo de cada produto em **uma única operação** (use `agg`).

### Pergunta 6
Em qual município o **preço da gasolina varia mais** (maior desvio padrão)? Retorne o nome do município, não o valor do desvio.

## Bloco 3, análise geográfica

### Pergunta 7
Quantas coletas existem em "SAO PAULO" (capital)? E quantos postos físicos distintos (CNPJs únicos) operam na capital?

### Pergunta 8
Liste os 5 municípios com **maior preço médio de gasolina**, do mais caro pro mais barato.

### Pergunta 9
Qual a bandeira mais frequente em SP capital? Quantos % das coletas da capital são dessa bandeira?

## Bloco 4, manipulação de strings

### Pergunta 10
Quantas coletas têm "AUTO POSTO" no nome da Revenda? Use `.str.contains`.

### Pergunta 11
Crie uma coluna `Inicial_Revenda` com a primeira letra do nome da Revenda. Depois, quantas coletas começam com a letra "A"?

## Bloco 5, análise temporal

### Pergunta 12
Crie a coluna `Mes` no formato `YYYY-MM` a partir de `Data da Coleta` (que está em `dd/mm/yyyy`). Em qual mês a gasolina foi mais cara em média?

## Bloco 6, merge e cálculo de margem (FINAL)

### Pergunta 13
Junte `postos` com `distrib` pela chave tripla `(Municipio, Produto, Mes)` usando `inner`. Quantas linhas tem o resultado? O número é maior ou menor que `postos`? Por quê?

### Pergunta 14
Crie a coluna `Margem` (Valor de Venda menos Preco_Distribuicao). Calcule a margem média por bandeira, do maior pro menor.

### Pergunta 15
Top 3 municípios com maior margem média. Qual o nome do município com a maior?

### Pergunta 16 (bônus)
Calcule a **proporção de postos sem bandeira** (BRANCA) em cada município. Liste os municípios onde mais de 30% das coletas são de postos BRANCA. Ordene do maior % pro menor.

---

# GABARITO

**SÓ OLHE DEPOIS DE TENTAR.** Se você só ler as soluções, não vai treinar nada.

Cada pergunta tem a resposta abaixo do enunciado. Em algumas, há comentário explicando a lógica.

### Gabarito 1
`postos.shape` e `distrib.shape` retornam tupla (linhas, colunas).

In [ ]:
print("postos:", postos.shape)
print("distrib:", distrib.shape)

### Gabarito 2
Se `Valor de Venda` viesse como `object`, seria porque o `decimal=","` não pegou. Aí qualquer operação numérica (mean, sum, max) falharia ou daria resultado errado.

In [ ]:
postos.head(10)
postos.info()

### Gabarito 3

In [ ]:
postos["CNPJ"] = postos["CNPJ"].str.strip()
postos["CNPJ"].nunique()

### Gabarito 4

In [ ]:
postos["Produto"].value_counts()

### Gabarito 5
Lista de funções dentro do `agg` aplica todas elas pra mesma coluna.

In [ ]:
postos.groupby("Produto")["Valor de Venda"].agg(["mean", "median", "min", "max"])

### Gabarito 6
`.idxmax()` retorna o RÓTULO (índice) onde o valor é máximo, ou seja, o nome do município.

In [ ]:
(postos[postos["Produto"] == "GASOLINA"]
    .groupby("Municipio")["Valor de Venda"]
    .std()
    .idxmax())

### Gabarito 7

In [ ]:
sp = postos[postos["Municipio"] == "SAO PAULO"]
print("coletas em SP capital:", len(sp))
print("postos físicos distintos:", sp["CNPJ"].nunique())

### Gabarito 8
Filtra gasolina, agrupa por município, tira média, ordena decrescente, pega 5 primeiros.

In [ ]:
(postos[postos["Produto"] == "GASOLINA"]
    .groupby("Municipio")["Valor de Venda"]
    .mean()
    .sort_values(ascending=False)
    .head(5))

### Gabarito 9
`normalize=True` dá proporção em vez de contagem absoluta.

In [ ]:
postos[postos["Municipio"] == "SAO PAULO"]["Bandeira"].value_counts(normalize=True).head(1)

### Gabarito 10
`.str.contains` retorna Series booleana. `.sum()` numa booleana conta quantos True existem.

In [ ]:
postos["Revenda"].str.contains("AUTO POSTO").sum()

### Gabarito 11

In [ ]:
postos["Inicial_Revenda"] = postos["Revenda"].str[0]
len(postos[postos["Inicial_Revenda"] == "A"])

### Gabarito 12
Truque: na string `dd/mm/yyyy`, o ano está nas posições 6:10 e o mês nas 3:5. Concatena com hífen.

In [ ]:
postos["Mes"] = postos["Data da Coleta"].str[6:10] + "-" + postos["Data da Coleta"].str[3:5]
(postos[postos["Produto"] == "GASOLINA"]
    .groupby("Mes")["Valor de Venda"]
    .mean()
    .idxmax())

### Gabarito 13
O resultado deve ser **próximo ao tamanho de postos**, porque cada linha de postos casa com uma de distrib (quando a combinação Municipio+Produto+Mes existe nos dois). Se ficar gigante, é cartesiano (chave incompleta). Se ficar muito menor, alguma chave não bate (formato de mês diferente, por exemplo).

In [ ]:
juntos = pd.merge(
    postos, distrib,
    on=["Municipio", "Produto", "Mes"],
    how="inner",
)
print("juntos:", juntos.shape)
print("postos:", postos.shape)

### Gabarito 14

In [ ]:
juntos["Margem"] = juntos["Valor de Venda"] - juntos["Preco_Distribuicao"]
juntos.groupby("Bandeira")["Margem"].mean().sort_values(ascending=False)

### Gabarito 15

In [ ]:
top = juntos.groupby("Municipio")["Margem"].mean().sort_values(ascending=False).head(3)
print(top)
print("\nMaior:", top.idxmax())

### Gabarito 16
Lógica: pra cada município, conta a proporção de coletas com bandeira BRANCA. Pode fazer via `value_counts(normalize=True)` agrupado, ou via cálculo manual: cria coluna booleana e tira média (média de booleana = proporção de True).

In [ ]:
postos["eh_branca"] = postos["Bandeira"] == "BRANCA"
prop = postos.groupby("Municipio")["eh_branca"].mean()
prop[prop > 0.30].sort_values(ascending=False)

---

## Pontuação sugerida

Se você acertou:
- **0 a 5**: Revisa o cheat sheet inteiro, principalmente filtro lógico e groupby.
- **6 a 10**: Tá no caminho. Foco em merge e cenários combinados.
- **11 a 13**: Pronto pra ir.
- **14 a 16**: Tá sobrando preparo.